In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# การจำลองแบบแม่นยำด้วย Qiskit SDK primitives

<details>
<summary><b>เวอร์ชันแพ็กเกจ</b></summary>

โค้ดในหน้านี้พัฒนาโดยใช้ข้อกำหนดต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.3.0
```
</details>
Reference primitives ใน Qiskit SDK ทำการจำลอง statevector แบบ local โดยการจำลองเหล่านี้ไม่รองรับการสร้างแบบจำลองสัญญาณรบกวนของอุปกรณ์ แต่มีประโยชน์สำหรับการสร้างต้นแบบอัลกอริธึมอย่างรวดเร็วก่อนที่จะศึกษาเทคนิคการจำลองขั้นสูง ([โดยใช้ Qiskit Aer](./simulate-stabilizer-circuits)) หรือรันบนอุปกรณ์จริง ([Qiskit Runtime primitives](primitives))

Estimator primitive สามารถคำนวณค่าความคาดหวังของวงจรได้ และ Sampler primitive สามารถสุ่มตัวอย่างจากการกระจายผลลัพธ์ของวงจรได้

ส่วนต่อไปนี้แสดงวิธีใช้ reference primitives เพื่อรัน workflow ของคุณในเครื่อง

## ใช้งาน reference Estimator
การใช้งาน reference ของ `EstimatorV2` ใน `qiskit.primitives` ที่รันบน local statevector simulators คือคลาส [`StatevectorEstimator`](../api/qiskit/qiskit.primitives.StatevectorEstimator) มันรับ Circuit, observable และ parameter เป็น input และคืนค่าความคาดหวังที่คำนวณในเครื่อง

โค้ดต่อไปนี้เตรียม input ที่จะใช้ในตัวอย่างที่ตามมา ประเภท input ที่รองรับสำหรับ observable คือ [`qiskit.quantum_info.SparsePauliOp`](../api/qiskit/qiskit.quantum_info.SparsePauliOp) โปรดสังเกตว่า Circuit ในตัวอย่างมีการกำหนดค่าพารามิเตอร์ แต่คุณสามารถรัน Estimator กับ Circuit ที่ไม่มีพารามิเตอร์ได้เช่นกัน

> **Note:** Circuit ที่ส่งให้ Estimator จะต้อง**ไม่**มีการ**วัดผล**ใด ๆ

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

# circuit for which you want to obtain the expected value
circuit = QuantumCircuit(2)
circuit.ry(Parameter("theta"), 0)
circuit.h(0)
circuit.cx(0, 1)
circuit.draw("mpl", style="iqp")

<Image src="../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/5b41a52d-8f15-4ce4-b3f6-effd91946d9c-0.svg" alt="Output of the previous code cell" />

In [2]:
from qiskit.quantum_info import SparsePauliOp
import numpy as np

# observable(s) whose expected values you want to compute

observable = SparsePauliOp(["II", "XX", "YY", "ZZ"], coeffs=[1, 1, -1, 1])

# value(s) for the circuit parameter(s)
parameter_values = [[0], [np.pi / 6], [np.pi / 2]]

> **Tip:** ขั้นตอนการทำงานของ Qiskit Runtime primitives ต้องการให้ Circuit และ observable ถูกแปลงให้ใช้เฉพาะคำสั่งที่รองรับโดย QPU (เรียกว่า *instruction set architecture (ISA)* circuits และ observables) Reference primitives ยังคงรับคำสั่งแบบ abstract ได้ เนื่องจากพึ่งพาการจำลอง statevector แบบ local แต่การ transpile Circuit อาจยังเป็นประโยชน์ในแง่ของการปรับแต่ง Circuit
> 
>   ```python
>   # Generate a pass manager without providing a backend
>   from qiskit.transpiler import generate_preset_pass_manager
> 
>   pm = generate_preset_pass_manager(optimization_level=1)
>   isa_circuit = pm.run(circuit)
>   isa_observable = observable.apply_layout(isa_circuit.layout)
>   ```

### Initialize Estimator
สร้าง instance ของ [`qiskit.primitives.StatevectorEstimator`](../api/qiskit/qiskit.primitives.StatevectorEstimator)

In [3]:
from qiskit.primitives import StatevectorEstimator

estimator = StatevectorEstimator()

### รันและรับผลลัพธ์
ตัวอย่างนี้ใช้ Circuit เดียว (ประเภท [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit)) และ observable เดียว

รันการประมาณโดยเรียกใช้เมธอด [`StatevectorEstimator.run`](../api/qiskit/qiskit.primitives.StatevectorEstimator#run) ซึ่งคืนค่า instance ของออบเจ็กต์ [`PrimitiveJob`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveJob) คุณสามารถรับผลลัพธ์จาก job (เป็นออบเจ็กต์ [`qiskit.primitives.PrimitiveResult`](../api/qiskit/qiskit.primitives.PrimitiveResult)) ด้วยเมธอด [`qiskit.primitives.PrimitiveJob.result`](../api/qiskit/qiskit.primitives.PrimitiveJob#result)

In [4]:
job = estimator.run([(circuit, observable, parameter_values)])
result = job.result()
print(f" > Result class: {type(result)}")

 > Result class: <class 'qiskit.primitives.containers.primitive_result.PrimitiveResult'>


#### Get the expected value from the result

The primitives result outputs an array of [`PubResult`](/docs/api/qiskit/qiskit.primitives.PubResult#pubresult) objects, where each item of the array is a `PubResult` object that contains in its data the array of evaluations corresponding to every circuit-observable combination in the PUB.

To retrieve the expectation values and metadata for the first (and in this case, only) circuit evaluation, we must access the evaluation [`data`](/docs/api/qiskit/qiskit.primitives.PubResult#data) for PUB 0:

In [5]:
print(f" > Expectation value: {result[0].data.evs}")
print(f" > Metadata: {result[0].metadata}")

 > Expectation value: [4.         3.73205081 2.        ]
 > Metadata: {'target_precision': 0.0, 'circuit_metadata': {}}


#### รับค่าความคาดหวังจากผลลัพธ์
ผลลัพธ์ของ primitives จะส่งออกเป็น array ของออบเจ็กต์ [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#pubresult) โดยแต่ละรายการใน array คือออบเจ็กต์ `PubResult` ที่มีอยู่ใน data ของ array การประเมินที่สอดคล้องกับทุกคู่ Circuit-observable ใน PUB

ในการดึงค่าความคาดหวังและ metadata สำหรับการประเมิน Circuit แรก (และในกรณีนี้เพียงอันเดียว) เราต้องเข้าถึง [`data`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#data) การประเมินสำหรับ PUB 0:

In [6]:
# Estimate expectation values for two PUBs, both with 0.05 precision.
precise_job = estimator.run(
    [(circuit, observable, parameter_values)], precision=0.05
)

For a full example, see the [Primitives examples](primitives-examples#estimator-examples) page.

## Use the reference Sampler

The reference implementations of `SamplerV2` in `qiskit.primitives` is the [`StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler) class. It takes circuits and parameters as inputs and returns the results from sampling from the output probability distributions as a quasi-probability distribution of output states.

The following code prepares the inputs used in the examples that follow. Note that
these examples run a single parametrized circuit, but you can also run the Sampler
on non-parametrized circuits.

In [7]:
from qiskit import QuantumCircuit

circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw("mpl", style="iqp")

<Image src="../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/d4c0ac3b-8e5b-4cde-bb26-256324982c2c-0.svg" alt="Output of the previous code cell" />

### กำหนดตัวเลือกการรัน Estimator
โดยค่าเริ่มต้น reference Estimator จะทำการคำนวณ statevector แบบแม่นยำโดยอิงจากคลาส [`quantum_info.Statevector`](../api/qiskit/qiskit.quantum_info.Statevector)
อย่างไรก็ตาม สามารถปรับแต่งเพื่อนำผลกระทบของ sampling overhead มาใช้ (หรือที่เรียกว่า "shot noise")

Estimator รับ argument `precision` ที่แสดงถึง error bar ที่การใช้งาน primitive ควรมุ่งหมายสำหรับการประมาณค่าความคาดหวัง นี่คือ sampling overhead และกำหนดเฉพาะในเมธอด `.run()` เท่านั้น ซึ่งช่วยให้คุณปรับแต่งตัวเลือกได้ถึงระดับ PUB

In [8]:
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()

สำหรับตัวอย่างฉบับสมบูรณ์ ดูหน้า [ตัวอย่าง Primitives](primitives-examples#estimator-examples)

## ใช้งาน reference Sampler
การใช้งาน reference ของ `SamplerV2` ใน `qiskit.primitives` คือคลาส [`StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler) มันรับ Circuit และ parameter เป็น input และคืนผลลัพธ์จากการสุ่มตัวอย่างจากการกระจายความน่าจะเป็นของผลลัพธ์เป็น quasi-probability distribution ของ state ผลลัพธ์

โค้ดต่อไปนี้เตรียม input ที่ใช้ในตัวอย่างที่ตามมา โปรดสังเกตว่าตัวอย่างเหล่านี้รัน Circuit ที่มีพารามิเตอร์เพียงวงจรเดียว แต่คุณยังสามารถรัน Sampler กับ Circuit ที่ไม่มีพารามิเตอร์ได้

In [9]:
# execute 1 circuit with Sampler
job = sampler.run([circuit])
pub_result = job.result()[0]
print(f" > Result class: {type(pub_result)}")

 > Result class: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


![Output of the previous code cell](../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/d4c0ac3b-8e5b-4cde-bb26-256324982c2c-0.svg)

> **Note:** วงจรควอนตัมทุกวงจรที่ส่งให้ Sampler **ต้อง**มีการวัดผล

> **Tip:** ขั้นตอนการทำงานของ Qiskit Runtime primitives ต้องการให้ Circuit ถูกแปลงให้ใช้เฉพาะคำสั่งที่รองรับโดย QPU (เรียกว่า ISA circuits) Reference primitives ยังคงรับคำสั่งแบบ abstract ได้ เนื่องจากพึ่งพาการจำลอง statevector แบบ local แต่การ transpile Circuit อาจยังเป็นประโยชน์ในแง่ของการปรับแต่ง Circuit
> 
>   ```python
>   # Generate a pass manager without providing a backend
>   from qiskit.transpiler import generate_preset_pass_manager
> 
>   pm = generate_preset_pass_manager(optimization_level=1)
>   isa_circuit = pm.run(qc)
>   ```

### Initialize `SamplerV2`
สร้าง instance ของ [`qiskit.primitives.StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler):

In [10]:
from qiskit.transpiler import generate_preset_pass_manager

# create two circuits
circuit1 = circuit.copy()
circuit2 = circuit.copy()

# transpile circuits
pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit1 = pm.run(circuit1)
isa_circuit2 = pm.run(circuit2)
# execute 2 circuits using Sampler
job = sampler.run([(isa_circuit1), (isa_circuit2)])
pub_result_1 = job.result()[0]
pub_result_2 = job.result()[1]
print(f" > Result class: {type(pub_result)}")

 > Result class: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


### รันและรับผลลัพธ์

In [11]:
# Define quantum circuit with 2 qubits
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw()

        ┌───┐      ░ ┌─┐   
   q_0: ┤ H ├──■───░─┤M├───
        └───┘┌─┴─┐ ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░──╫─┤M├
             └───┘ ░  ║ └╥┘
meas: 2/══════════════╩══╩═
                      0  1 

In [12]:
# Transpile circuit
pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit = pm.run(circuit)
# Run using sampler
result = sampler.run([circuit]).result()
# Access result data for PUB 0
data_pub = result[0].data
# Access bitstring for the classical register "meas"
bitstrings = data_pub.meas.get_bitstrings()
print(f"The number of bitstrings is: {len(bitstrings)}")
# Get counts for the classical register "meas"
counts = data_pub.meas.get_counts()
print(f"The counts are: {counts}")

The number of bitstrings is: 1024
The counts are: {'11': 515, '00': 509}


Primitives รับ PUB หลายรายการเป็น input และแต่ละ PUB จะได้รับผลลัพธ์ของตัวเอง ดังนั้นคุณสามารถรัน Circuit ต่าง ๆ ด้วยการรวมกันของ parameter/observable ต่าง ๆ และดึงผลลัพธ์ PUB:

In [13]:
# Sample two circuits at 128 shots each.
sampler.run([isa_circuit1, isa_circuit2], shots=128)
# Sample two circuits at different amounts of shots. The "None"s are necessary
# as placeholders
# for the lack of parameter values in this example.
sampler.run([(isa_circuit1, None, 123), (isa_circuit2, None, 456)])

For a full example, see the [Primitives examples](./primitives-examples#sampler-examples) page.
## Next steps

<Admonition type="tip" title="Recommendations">
  - For higher-performance simulation that can handle larger circuits, or to incorporate noise models into your simulation, see [Exact and noisy simulation with Qiskit Aer primitives](simulate-with-qiskit-aer).
  - To learn how to use Quantum Composer for simulation, see the [IBM Quantum Composer](/docs/guides/composer) guide.
  - Read the [Qiskit Estimator API](/docs/api/qiskit/1.4/qiskit.primitives.Estimator) reference.
  - Read the [Qiskit Sampler API](/docs/api/qiskit/1.4/qiskit.primitives.Sampler) reference.
  - Read [Migrate to V2 primitives](/docs/guides/v2-primitives).
</Admonition>